In [ ]:
os.chdir('D:\\WEI\\軟體\\TWStock')

In [2]:
import os
import numpy as np
data_date = '2024-01-19'  ## 手動輸出

## 讀取 SITB
patho = os.getcwd()
path = patho + '\\SITB_rank\\' + data_date
os.chdir(path)
TWSE_SITB = np.load('TWSE_SITB_rank.npy')
TPEX_SITB = np.load('TPEX_SITB_rank.npy')
os.chdir(patho)

twse_code = TWSE_SITB
tpex_code = TPEX_SITB

print('上市公司: ' + str(len(twse_code)))
print('上櫃公司: ' + str(len(tpex_code)))
# print(twse_code,tpex_code)

上市公司: 112
上櫃公司: 19


In [3]:
## 抓取股價、量
## 注意!! output時間序顛倒
from FinMind.data import DataLoader
dl = DataLoader()
def stock_crawler(stock,start,end):
    ##==========================##
    # input
    # stock : 股票代碼
    # start : 擷取股市資料起始日期
    # end : 擷取股市資料終止日期
    try:
        stock_inf = dl.taiwan_stock_daily(stock, start, end)

        date = list(stock_inf.date)[::-1]
        price = list(stock_inf.close)[::-1]           ## 擷取收盤價
        volume = list((stock_inf.Trading_Volume)/1000)[::-1]  ## 擷取成交量
        return price,volume
    except:
        print('Crawlering error')

In [4]:
## 計算個股價、量、ma (計算近20個交易日)
## 注意!! output時間序顛倒
import pandas_datareader as pdr
def ma_count(price,volume):
    ##==========================##
    # input

    stock_reslut = []
    ## 計算近20個交易日
    for i in range(20):
        p5 = []
        p10 = []
        p20 = []
        p60 = []
        v5 = []
        v10 =[]

        for j in range(5):
            j = i + j
            p5.append(price[j])
            v5.append(volume[j])
        for j in range(10):
            j = i + j
            p10.append(price[j])
            v10.append(volume[j])
        for j in range(20):
            j = i + j
            p20.append(price[j])
        for j in range(60):
            j = i + j
            p60.append(price[j])

        price_n = price[i]
        volume_n = int(volume[i])
        ma5 = sum(p5)/5
        ma10 = sum(p10)/10
        ma20 = sum(p20)/20
        ma60 = sum(p60)/60
        vma5 = sum(v5)/5
        vma10 = sum(v10)/10

        stock_r = [price_n,volume_n,ma5,ma10,ma20,ma60,vma5,vma10]
        stock_reslut.append(stock_r)

    return stock_reslut

## 注意!! output時間序顛倒

In [5]:
## 檢視價、量、ma關係
## 注意!! output時間序顛倒
def stock_check(stock_reslut):
    
    check_result = []
    ## 檢視當天股價與ma之關係，以及成交量與vma之關係，最後檢視近10日平均成交量是否大於500張
    for i in stock_reslut:
        stock_n = i            #個股當天資訊list
        price_n = stock_n[0]   #當天股價
        volume_n = stock_n[1]  #當天成交量
        ma5 = stock_n[2]       #當天5ma
        ma10 = stock_n[3]      #當天10ma
        ma20 = stock_n[4]      #當天20ma
        ma60 = stock_n[5]      #當天60ma
        vma5 = stock_n[6]      #當天5vma
        vma10 = stock_n[7]     #當天10vma
        
        chk_ma5 = (price_n - ma5)/ma5 * 100      #與均線距離(%) ma5
        chk_ma10 = (price_n - ma10)/ma10 * 100   #與均線距離(%) ma10
        chk_ma20 = (price_n - ma20)/ma20 * 100   #與均線距離(%) ma20
        chk_ma60 = (price_n - ma60)/ma60 * 100   #與均線距離(%) ma60
        
        if vma5 == 0:
            vma5 = 1
        if vma10 == 0:
            vma10 = 1
        chk_vma5 = (volume_n - vma5)/vma5 * 100     #與成交量均線距離(%) vma5
        chk_vma10 = (volume_n - vma10)/vma10 * 100  #與成交量均線距離(%) vma10
        
        check_r = [price_n,volume_n,chk_ma5,chk_ma10,chk_ma20,chk_ma60,chk_vma5,chk_vma10]
        check_result.append(check_r)
    return check_result

In [6]:
### 主程式 ### (該程式碼設定近130天內所有交易日)
## 設定爬蟲時間段
import datetime
d = int(input('要抓幾天前的資料勒？(當天=0): '))
start = datetime.date.today() - datetime.timedelta(days=130)
end = datetime.date.today() - datetime.timedelta(days=d)
print('請確認起始日期：' + str(end))

twse_stock_result = {}
twse_check_result = {}
tpex_stock_result = {}
tpex_check_result = {}

請確認起始日期：2024-01-19


In [7]:
### 主程式 ### (該程式碼設定近150天內所有交易日)
import time

error = []
## 上市公司全部成果(價、量、ma)
for i in twse_code[:2]:
    if i not in list(twse_check_result.keys()):
        ## 爬蟲
        try:
            price,volume = stock_crawler(i,start,end)
            time.sleep(1)
        except:
            error.append(i)
            print('Crawler error : ' + 'TWSE--' + str(i))
            continue

        ## 計算
        try:
            stock_result = ma_count(price,volume)
            twse_stock_result[i] = stock_result
            check_result = stock_check(stock_result)
            twse_check_result[i] = check_result
        except:
            error.append(i)
            print('Processing error : ' + 'TWSE--' + str(i))

## 上櫃公司全部成果(價、量、ma)
for i in tpex_code[:2]:
    if i not in list(tpex_check_result.keys()):
        ## 爬蟲
        try:
            price,volume = stock_crawler(i,start,end)
            time.sleep(1)
        except:
            error.append(i)
            print('Crawler error : ' + 'TPEX--' + str(i))
            continue

        ## 計算
        try:
            stock_result = ma_count(price,volume)
            tpex_stock_result[i] = stock_result
            check_result = stock_check(stock_result)
            tpex_check_result[i] = check_result
        except:
            error.append(i)
            print('Processing error : ' + 'TPEX--' + str(i))

if len(error) == 0:
    print('done.')
else:
    print('上市公司剩餘:' + str(len(twse_code) - len(twse_check_result)))
    print('上櫃公司剩餘:' + str(len(tpex_code) - len(tpex_check_result)))
    print('第' + str(j+1) + '次爬蟲結束!')
    print('Error:')
    print(error)

2024-01-21 23:25:17.163 | INFO     | FinMind.data.finmind_api:get_data:125 - download TaiwanStockPrice, data_id: 2382
2024-01-21 23:25:18.944 | INFO     | FinMind.data.finmind_api:get_data:125 - download TaiwanStockPrice, data_id: 2303
2024-01-21 23:25:20.393 | INFO     | FinMind.data.finmind_api:get_data:125 - download TaiwanStockPrice, data_id: 3131
2024-01-21 23:25:21.764 | INFO     | FinMind.data.finmind_api:get_data:125 - download TaiwanStockPrice, data_id: 3217


done.


In [8]:
print(twse_stock_result)

{'2382': [[240.0, 86067, 228.9, 224.75, 220.675, 209.75833333333333, 46394.206600000005, 40264.269199999995], [221.5, 34790, 227.1, 222.3, 219.45, 209.13333333333333, 39841.9468, 34612.8925], [228.5, 44081, 227.6, 222.05, 218.875, 208.95, 40046.8888, 33878.2292], [228.0, 23296, 225.5, 220.6, 217.625, 208.625, 35821.047000000006, 31792.0755], [226.5, 43734, 223.1, 219.2, 216.375, 208.16666666666666, 39218.2078, 33709.802299999996], [231.0, 53306, 220.6, 219.0, 215.175, 207.8, 34134.3318, 34922.9931], [224.0, 35815, 217.5, 217.6, 213.9, 207.425, 29383.8382, 31568.437599999997], [218.0, 22952, 216.5, 216.75, 212.75, 207.18333333333334, 27709.5696, 29696.750399999997], [216.0, 40282, 215.7, 216.55, 211.9, 207.24166666666667, 27763.104, 29112.9318], [214.0, 18315, 215.3, 216.5, 211.3, 207.33333333333334, 28201.3968, 27467.028700000003], [215.5, 29553, 217.4, 216.6, 210.75, 207.53333333333333, 35711.6544, 29493.853099999997], [219.0, 27444, 217.7, 216.6, 210.025, 207.90833333333333, 33753.03

In [36]:
## 儲存上市上櫃全部stock成果到電腦
import os
import numpy as np
patho = os.getcwd() 
os.chdir(os.getcwd() + '\stock_result')

pathw = os.getcwd() + '\\' + str(end)
if not os.path.isdir(pathw):
    os.mkdir(pathw)

os.chdir(pathw)
np.save('twse_stock_result',twse_stock_result)
np.save('tpex_stock_result',tpex_stock_result)
os.chdir(patho)

## 儲存上市上櫃全部check成果到電腦
patho = os.getcwd() 
os.chdir(os.getcwd() + '\check_result')

pathw = os.getcwd() + '\\' + str(end)
if not os.path.isdir(pathw):
    os.mkdir(pathw)

os.chdir(pathw)
np.save('twse_check_result',twse_check_result)
np.save('tpex_check_result',tpex_check_result)
os.chdir(patho)

In [ ]:
### 有bug，因 yahoo response資料格式改變 ###

# ## 抓取股價、量
# ## 注意!! output時間序顛倒
# import pandas_datareader as pdr
# def stock_crawler(stock,twse,start,end):
#     ##==========================##
#     # input
#     # stock : 股票代碼
#     # twse : 是否為上市公司。是(1)；不是(0)
#     # start : 擷取股市資料起始日期
#     # end : 擷取股市資料終止日期
    
#     if twse == 1: #是上市公司
#         stock_inf = pdr.DataReader(stock+'.TW', 'yahoo',start,end)
#     else:         #是上櫃公司
#         stock_inf = pdr.DataReader(stock+'.TWO', 'yahoo',start,end)

#     date = list(stock_inf.index)[::-1]
#     price = list(stock_inf['Close'])[::-1]           ## 擷取收盤價
#     volume = list((stock_inf['Volume'])/1000)[::-1]  ## 擷取成交量
    
    
#     ## 判斷擷取資料日期是否正確
#     date0 = str(date[0])[:10]
#     if date0 == str(end):
#         return price,volume
#     else:
#         print('Crawlering error')